# 02 Gold Layer: Customer Analytics Marts

This notebook builds the **Gold** layer from `data/silver`. Gold tables are analysis-ready marts for customer, product-line, and subscription-order analytics.

Run `01_silver_layer.ipynb` first.


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "data_cleaning":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "1.customer_transaction").exists():
    for parent in Path.cwd().parents:
        if (parent / "1.customer_transaction").exists():
            PROJECT_ROOT = parent
            break

DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
for folder in [DATA_DIR, SILVER_DIR, GOLD_DIR]:
    folder.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Silver directory:", SILVER_DIR)
print("Gold directory:", GOLD_DIR)

# Gold owns analytical marts. Remove stale Gold parquet files before rewriting.
for old_file in GOLD_DIR.glob("*.parquet"):
    old_file.unlink()


Project root: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project
Silver directory: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/silver
Gold directory: /Users/tohzhengfeng/Documents/SMU/Academia/MITB 2526 Apr/ISSS603 Customer Analytics/Project/data/gold


## 1. Shared Helpers


In [2]:
FX_RATES_TO_SGD = {
    "SG": 1.0,
    "MY": 1.0 / 3.30,  # 1 SGD = 3.30 MYR
    "HK": 1.0 / 6.10,  # 1 SGD = 6.10 HKD
}
ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="Asia/Singapore")

PRODUCT_MAP = {
    "lean-protein": "Lean Protein",
    "lean_protein": "Lean Protein",
    "clear-protein": "Clear Protein",
    "clear_protein": "Clear Protein",
    "collagen": "Collagen Glow",
    "soy-protein": "Soy Protein",
    "protein-bar": "Protein Bar",
    "shaker": "Accessories",
    "starter-kit": "Accessories",
    "multivitamin": "Supplements",
    "cap-": "Supplements",
}
MARKETPLACE_KEYWORDS = ["shopee", "lazada", "tokopedia", "redmart", "grab"]


def clean_id(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "nat"}:
        return pd.NA
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text


def is_array_value(value):
    return isinstance(value, (list, tuple, np.ndarray))


def clean_object_cols(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        non_null = df[col].dropna()
        if not non_null.empty and non_null.map(is_array_value).any():
            # Preserve list-like aggregation columns as parquet arrays instead of converting them to strings.
            df[col] = df[col].map(lambda x: list(x) if is_array_value(x) else ([] if pd.isna(x) or str(x).strip() == "" else [str(x).strip()]))
        else:
            df[col] = df[col].map(lambda x: pd.NA if pd.isna(x) or str(x).strip() == "" else str(x).strip())
    return df


def parse_numeric_cols(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def store_prefix(name):
    if pd.isna(name):
        return "Unknown"
    n = str(name).upper().replace("#", "").strip()
    if n.startswith("LPMY"):
        return "MY"
    if n.startswith("LPHK"):
        return "HK"
    if n.startswith("LPSG"):
        return "SG"
    if n.startswith("LP"):
        return "SG"
    return "Other"


def classify_product(handle):
    if pd.isna(handle):
        return "Unknown"
    h = str(handle).lower()
    for keyword, label in PRODUCT_MAP.items():
        if keyword in h:
            return label
    return "Other"


def classify_channel(row):
    tags = str(row.get("Tags", "") or "").lower()
    utm = str(row.get("Browser: UTM Source", "") or "").lower()
    name = str(row.get("Name", "") or "").lower()

    if any(keyword in tags for keyword in MARKETPLACE_KEYWORDS):
        return "Marketplace"
    if "subscription" in tags or "yotpo subscriptions" in tags or name.startswith("lpsg"):
        return "Subscription"
    if utm in {"facebook", "instagram", "tiktok"}:
        return "Paid Social"
    if utm in {"google", "bing"}:
        return "Paid Search"
    if utm == "affiliate":
        return "Affiliate"
    if utm in {"shopify_email", "email", "klaviyo"}:
        return "Email"
    return "Direct / Organic"


def save_parquet(df, folder, filename):
    path = folder / filename
    clean_df = clean_object_cols(df)
    clean_df.to_parquet(path, index=False)
    reloaded = pd.read_parquet(path)
    print(f"Saved {path.parent.name}/{filename}: {len(reloaded):,} rows, {len(reloaded.columns):,} columns")
    return path



def validation_summary(name, df, key_cols=None, expected_grain=None, required_cols=None, money_cols=None):
    print(f"\n[{name}] {len(df):,} rows x {len(df.columns):,} columns")
    if expected_grain:
        print(f" - expected grain: {expected_grain}")
    if required_cols:
        missing = [col for col in required_cols if col not in df.columns]
        print(f" - required columns present: {not missing}")
        if missing:
            raise AssertionError(f"{name} missing required columns: {missing}")
    if key_cols:
        missing_keys = [col for col in key_cols if col not in df.columns]
        if missing_keys:
            raise AssertionError(f"{name} missing key columns: {missing_keys}")
        key_nulls = df[key_cols].isna().any(axis=1).sum()
        duplicate_keys = df.duplicated(subset=key_cols).sum()
        print(f" - key columns: {key_cols}")
        print(f" - rows with null key: {key_nulls:,} ({key_nulls / max(len(df), 1):.1%})")
        print(f" - duplicate key rows: {duplicate_keys:,}")
    if money_cols:
        for col in [c for c in money_cols if c in df.columns]:
            non_numeric = pd.to_numeric(df[col], errors="coerce").isna() & df[col].notna()
            print(f" - money column {col}: non-numeric values {non_numeric.sum():,}")


## 2. Load Silver Tables

Gold reads only from Silver parquet files. It does not read the raw numbered folders.


In [3]:
required_silver = [
    "orders.parquet", "lines.parquet", "products.parquet", "recharge_orders.parquet",
]
missing = [name for name in required_silver if not (SILVER_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing Silver files. Run 01_silver_layer.ipynb first: {missing}")

orders = pd.read_parquet(SILVER_DIR / "orders.parquet")
lines = pd.read_parquet(SILVER_DIR / "lines.parquet")
products = pd.read_parquet(SILVER_DIR / "products.parquet")
recharge_orders = pd.read_parquet(SILVER_DIR / "recharge_orders.parquet")

print("Loaded Silver tables:")
for name, df in {
    "orders": orders,
    "lines": lines,
    "products": products,
    "recharge_orders": recharge_orders,
}.items():
    print(f" - {name}: {len(df):,} rows, {len(df.columns):,} columns")


Loaded Silver tables:
 - orders: 27,350 rows, 49 columns
 - lines: 50,963 rows, 42 columns
 - products: 56 rows, 25 columns
 - recharge_orders: 1,215 rows, 10 columns


## 3. Build Customer Mart

The customer mart is one row per Shopify customer and is designed for retention, RFM, repeat behavior, and CLV-style analysis.


In [4]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
customers = (
    orders.sort_values("order_date")
    .groupby("customer_id", as_index=False)
    .agg(
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        total_orders=("order_id", "count"),
        total_revenue_sgd=("order_revenue_sgd", "sum"),
        total_discount_sgd=("order_discount_sgd", "sum"),
        ever_subscribed=("is_subscription", "any"),
        ever_discounted=("has_discount", "any"),
        first_channel=("channel", "first"),
        first_product_category=("product_category", "first"),
        first_store=("store", "first"),
    )
)
second_orders = (
    orders.sort_values("order_date")
    .groupby("customer_id", as_index=False)
    .nth(1)[["customer_id", "order_date"]]
    .rename(columns={"order_date": "second_order_date"})
)
customers = customers.merge(second_orders, on="customer_id", how="left", validate="one_to_one")
customers["days_to_second"] = (customers["second_order_date"] - customers["first_order_date"]).dt.days
customers["is_repeat"] = customers["total_orders"] >= 2
customers["lifespan_days"] = (customers["last_order_date"] - customers["first_order_date"]).dt.days
customers["recency_days"] = (ANALYSIS_DATE - customers["last_order_date"]).dt.days
customers["cohort_month"] = customers["first_order_date"].dt.to_period("M").astype(str)

print("Customer rows:", f"{len(customers):,}")
print("Unique customer_id:", f"{customers['customer_id'].nunique():,}")
print("Repeat rate:", f"{customers['is_repeat'].mean():.1%}")


Customer rows: 13,780
Unique customer_id: 13,780
Repeat rate: 32.4%


## 4. Build Enriched Product-Line Mart

Line items are enriched with product master fields only through SKU matching. Missing product-master coverage is reported rather than patched through ambiguous joins.


In [5]:
line_merge_base = lines.copy()
line_merge_base["line_sku_clean"] = line_merge_base["Line: SKU"].map(clean_id)
product_lookup_cols = [
    col for col in ["product_variant_sku", "product_handle", "product_title", "vendor", "status", "variant_price", "cost_per_item", "product_category"]
    if col in products.columns
]
product_lookup = products[product_lookup_cols].copy()

order_lines_enriched = line_merge_base.merge(
    product_lookup,
    left_on="line_sku_clean",
    right_on="product_variant_sku",
    how="left",
    validate="many_to_one",
    suffixes=("", "_product_master"),
)
matched_product = order_lines_enriched["product_variant_sku"].notna()
print("Line items with SKU:", f"{line_merge_base['line_sku_clean'].notna().mean():.1%}")
print("Line items matched to product master by SKU:", f"{matched_product.mean():.1%}")
print("Top unmatched line SKUs:")
print(order_lines_enriched.loc[~matched_product, "line_sku_clean"].value_counts(dropna=False).head(10).to_string())


Line items with SKU: 51.6%
Line items matched to product master by SKU: 11.1%
Top unmatched line SKUs:
line_sku_clean
NaN              24680
0724999808361     1860
0724999807814     1494
0724999807821     1206
0724999808507     1168
0724999807807     1067
0724999807999      844
0724999810463      796
0724999807937      769
0724999810470      704


## 5. Build Validated Recharge Order Mart

Only Recharge orders are enriched with Shopify order context, and only through the clear `shopify_order_id -> order_id` key. Other Recharge files stay in Silver because checkout items, recurring items, churned subscriptions, and reactivations have different grains.


In [6]:
def try_enrich_recharge_orders(recharge_df, orders_df):
    print("Recharge -> Shopify merge validation")
    if "shopify_order_id" not in recharge_df.columns or "order_id" not in orders_df.columns:
        print(" - required key missing; enriched table will not be created")
        return None

    right_duplicates = orders_df["order_id"].duplicated().sum()
    left_key_coverage = recharge_df["shopify_order_id"].notna().mean()
    print(" - left rows before merge:", f"{len(recharge_df):,}")
    print(" - Shopify order_id duplicates:", f"{right_duplicates:,}")
    print(" - Recharge shopify_order_id coverage:", f"{left_key_coverage:.1%}")
    if right_duplicates > 0:
        print(" - Shopify order_id is not unique; merge blocked")
        return None

    order_context_cols = [
        "order_id", "customer_id", "order_date", "store", "channel", "product_category",
        "order_revenue_sgd", "shipping_revenue_sgd", "order_total_incl_shipping_sgd", "order_discount_sgd", "is_subscription", "source_file",
    ]
    order_context = orders_df[[col for col in order_context_cols if col in orders_df.columns]].copy()
    order_context = order_context.rename(columns={
        "customer_id": "shopify_customer_id",
        "order_date": "shopify_order_date",
        "store": "shopify_store",
        "channel": "shopify_channel",
        "product_category": "shopify_first_product_category",
        "order_revenue_sgd": "shopify_revenue_excl_shipping_sgd",
        "shipping_revenue_sgd": "shopify_shipping_sgd",
        "order_total_incl_shipping_sgd": "shopify_total_incl_shipping_sgd",
        "order_discount_sgd": "shopify_discount_sgd",
        "is_subscription": "shopify_is_subscription_tag",
        "source_file": "shopify_source_file",
    })

    merged = recharge_df.merge(
        order_context,
        left_on="shopify_order_id",
        right_on="order_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    unmatched_rate = (merged["_merge"] == "left_only").mean()
    print(" - rows after merge:", f"{len(merged):,}")
    print(" - row-count unchanged:", len(merged) == len(recharge_df))
    print(" - unmatched Recharge orders:", f"{unmatched_rate:.1%}")
    print(" - merge status counts:")
    print(merged["_merge"].value_counts().to_string())

    if len(merged) != len(recharge_df):
        print(" - row-count changed unexpectedly; enriched table will not be saved")
        return None
    return merged.drop(columns=["_merge"])

recharge_orders_enriched = try_enrich_recharge_orders(recharge_orders, orders)
if recharge_orders_enriched is None:
    print("No Recharge enriched table created.")
else:
    print("Recharge enriched table is valid at the Recharge-order grain.")


Recharge -> Shopify merge validation
 - left rows before merge: 1,215
 - Shopify order_id duplicates: 0
 - Recharge shopify_order_id coverage: 100.0%
 - rows after merge: 1,215
 - row-count unchanged: True
 - unmatched Recharge orders: 4.3%
 - merge status counts:
_merge
both          1163
left_only       52
right_only       0
Recharge enriched table is valid at the Recharge-order grain.


## 6. Save Gold Tables And README


In [7]:
gold_outputs = {
    "customers.parquet": customers,
    "order_lines_enriched.parquet": order_lines_enriched,
}
if recharge_orders_enriched is not None:
    gold_outputs["recharge_orders_enriched.parquet"] = recharge_orders_enriched

saved_paths = [save_parquet(df, GOLD_DIR, filename) for filename, df in gold_outputs.items()]

gold_readme = """# Gold Layer

Gold contains analysis-ready marts built from validated Silver tables. These are the recommended starting point for customer analytics, dashboards, and modeling.

| Dataset | Grain | What it contains | Notes |
|---|---:|---|---|
| `customers.parquet` | 1 row per Shopify customer | First/last order date, total orders, non-shipping revenue, discounts, repeat flag, subscription/discount flags, first channel/product/store, recency, cohort month | Recommended for retention, RFM, repeat behavior, and CLV-style analysis. Revenue uses `order_revenue_sgd`, excluding shipping. |
| `order_lines_enriched.parquet` | 1 row per Shopify product line item | Silver line items enriched with product master fields where SKU matching is valid; line-level values exclude shipping | Product-master coverage is reported in the notebook; unmatched older SKUs are kept rather than forced into bad joins. |
| `recharge_orders_enriched.parquet` | 1 row per Recharge order | Recharge orders enriched with Shopify order context using validated `shopify_order_id -> order_id` many-to-one merge | Created only when merge validation passes. Checkout, recurring, churn, and reactivation files remain separate because they have different grains. |

## Recommended Usage

Use `customers.parquet` for customer-level questions, `order_lines_enriched.parquet` for product/category/basket questions, and `recharge_orders_enriched.parquet` for subscription order questions. Use Silver only when you need source-level auditability or a different custom aggregation.
"""
(GOLD_DIR / "README.md").write_text(gold_readme, encoding="utf-8")
print("Saved README:", (GOLD_DIR / "README.md").relative_to(PROJECT_ROOT))


Saved gold/customers.parquet: 13,780 rows, 17 columns
Saved gold/order_lines_enriched.parquet: 50,963 rows, 51 columns
Saved gold/recharge_orders_enriched.parquet: 1,215 rows, 22 columns
Saved README: data/gold/README.md


## 7. Gold Validation


In [8]:
print("Gold validation checks")

validation_summary(
    "customers",
    customers,
    key_cols=["customer_id"],
    expected_grain="one row per Shopify customer",
    required_cols=["customer_id", "first_order_date", "last_order_date", "total_orders", "total_revenue_sgd", "is_repeat", "cohort_month"],
    money_cols=["total_revenue_sgd", "total_discount_sgd"],
)
validation_summary(
    "order_lines_enriched",
    order_lines_enriched,
    key_cols=["order_id", "line_id"],
    expected_grain="one row per Shopify product line item enriched with product master context",
    required_cols=["order_id", "line_id", "line_sku_clean", "Line: Total"],
    money_cols=["Line: Price", "Line: Discount", "Line: Total", "variant_price", "cost_per_item"],
)
if recharge_orders_enriched is not None:
    validation_summary(
        "recharge_orders_enriched",
        recharge_orders_enriched,
        key_cols=["recharge_order_id"],
        expected_grain="one row per Recharge order enriched with Shopify order context",
        required_cols=["recharge_order_id", "shopify_order_id", "shopify_revenue_excl_shipping_sgd", "shopify_shipping_sgd"],
        money_cols=["order_total", "order_gross_revenue", "shopify_revenue_excl_shipping_sgd", "shopify_shipping_sgd", "shopify_total_incl_shipping_sgd"],
    )

assert customers["customer_id"].is_unique, "gold/customers.parquet must be one row per customer_id"
assert len(order_lines_enriched) == len(lines), "product enrichment must not change line-item grain"
assert "order_revenue_sgd" in orders.columns, "Gold must use non-shipping order revenue from Silver"
assert np.isclose(customers["total_revenue_sgd"].sum(), orders["order_revenue_sgd"].sum(), rtol=0, atol=0.01), "customer revenue must reconcile to non-shipping Silver order revenue"
if recharge_orders_enriched is not None:
    assert len(recharge_orders_enriched) == len(recharge_orders), "Recharge enrichment must not change Recharge order grain"
assert (GOLD_DIR / "README.md").exists(), "Gold README missing"

print("\nGold validation passed.")
print("\nAll Gold parquet files can be read back:")
for path in saved_paths:
    df = pd.read_parquet(path)
    print(f" - {path.relative_to(PROJECT_ROOT)}: {len(df):,} rows")

Gold validation checks

[customers] 13,780 rows x 17 columns
 - expected grain: one row per Shopify customer
 - required columns present: True
 - key columns: ['customer_id']
 - rows with null key: 0 (0.0%)
 - duplicate key rows: 0
 - money column total_revenue_sgd: non-numeric values 0
 - money column total_discount_sgd: non-numeric values 0

[order_lines_enriched] 50,963 rows x 51 columns
 - expected grain: one row per Shopify product line item enriched with product master context
 - required columns present: True
 - key columns: ['order_id', 'line_id']
 - rows with null key: 0 (0.0%)
 - duplicate key rows: 0
 - money column Line: Price: non-numeric values 0
 - money column Line: Discount: non-numeric values 0
 - money column Line: Total: non-numeric values 0
 - money column variant_price: non-numeric values 0
 - money column cost_per_item: non-numeric values 0

[recharge_orders_enriched] 1,215 rows x 22 columns
 - expected grain: one row per Recharge order enriched with Shopify orde

# 8. Data Preview

In [9]:
customers_df = pd.read_parquet(GOLD_DIR / "customers.parquet")
print(customers_df.shape)
customers_df.head()

(13780, 17)


,customer_id,first_order_date,last_order_date,total_orders,total_revenue_sgd,total_discount_sgd,ever_subscribed,ever_discounted,first_channel,first_product_category,first_store,second_order_date,days_to_second,is_repeat,lifespan_days,recency_days,cohort_month
0,6327387259135,2022-06-29 00:00:00+08:00,2022-06-29 00:00:00+08:00,1,1693.12,0.00,False,False,Subscription,Unknown,SG,NaT,NaN,False,0,1401,2022-06
1,6327388733695,2021-10-04 00:00:00+08:00,2024-05-03 00:00:00+08:00,2,336.01,5.99,False,True,Subscription,Unknown,SG,2024-05-03 00:00:00+08:00,942.0,True,942,727,2021-10
2,6327388799231,2021-02-05 00:00:00+08:00,2021-11-05 00:00:00+08:00,3,88.17,0.00,False,False,Subscription,Unknown,SG,2021-10-11 00:00:00+08:00,248.0,True,273,1637,2021-02
3,6327389520127,2021-03-04 00:00:00+08:00,2021-03-04 00:00:00+08:00,1,29.00,0.00,False,False,Subscription,Unknown,SG,NaT,NaN,False,0,1883,2021-03
4,6327390699775,2020-08-18 00:00:00+08:00,2020-11-29 00:00:00+08:00,2,492.98,0.00,False,False,Subscription,Unknown,SG,2020-11-29 00:00:00+08:00,103.0,True,103,1978,2020-08


In [10]:
order_lines_enriched_df = pd.read_parquet(GOLD_DIR / "order_lines_enriched.parquet")
print(order_lines_enriched_df.shape)
order_lines_enriched_df.head()

(50963, 51)


,order_id,customer_id,order_date,processed_at_sgt,store,Currency,Payment: Status,Order Fulfillment Status,Shipping: Zip,Shipping: City,Shipping: Province,Shipping: Country,Shipping: Country Code,line_id,Line: Type,Line: Product ID,Line: Product Handle,Line: Title,Line: Name,Line: Variant ID,Line: Variant Title,Line: SKU,Line: Quantity,Line: Price,Line: Discount,Line: Discount Allocation,Line: Discount per Item,Line: Total,Line: Grams,Line: Requires Shipping,Line: Vendor,Line: Gift Card,Line: Variant SKU,Line: Variant Barcode,Line: Variant Weight,Line: Variant Weight Unit,Line: Variant Inventory Qty,Line: Variant Cost,Line: Variant Price,source_file,source_row_number,product_category,line_sku_clean,product_variant_sku,product_handle,product_title,vendor,status,variant_price,cost_per_item,product_category_product_master
0,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675926271,Line Item,NaN,NaN,"Better Whey - 500g Pack, Chocolate Dinosaur","Better Whey - 500g Pack, Chocolate Dinosaur",NaN,NaN,NaN,1.0,22.15,0.0,0.0,0.0,22.15,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,2,Unknown,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
1,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675959039,Line Item,NaN,NaN,Green Tea Extract Capsules - 60 Capsules,Green Tea Extract Capsules - 60 Capsules,NaN,NaN,NaN,1.0,13.50,0.0,0.0,0.0,13.50,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,3,Unknown,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
2,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664675991807,Line Item,NaN,NaN,"Better Whey - 500g Pack, Mango","Better Whey - 500g Pack, Mango",NaN,NaN,NaN,1.0,22.15,0.0,0.0,0.0,22.15,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,4,Unknown,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
3,4992746586367,6423777902847,2020-12-31 00:00:00+08:00,2020-12-31 03:27:50+08:00,SG,SGD,paid,fulfilled,461055,Singapore,NaN,Singapore,SG,12664676024575,Line Item,NaN,NaN,L-Carnitine,L-Carnitine,NaN,NaN,NaN,1.0,25.50,0.0,0.0,0.0,25.50,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,5,Unknown,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
4,4992746619135,6328286576895,2020-12-29 00:00:00+08:00,2020-12-29 09:59:53+08:00,SG,SGD,paid,fulfilled,328358,Singapore,NaN,Singapore,SG,12664676122879,Line Item,NaN,NaN,"Better Whey - 1KG Pack, Vanilla","Better Whey - 1KG Pack, Vanilla",NaN,NaN,NaN,1.0,39.10,0.0,0.0,0.0,39.10,0.0,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1_1.orders-2020_20260505.xlsx,11,Unknown,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN


In [11]:
recharge_orders_enriched_df = pd.read_parquet(GOLD_DIR / "recharge_orders_enriched.parquet")
print(recharge_orders_enriched_df.shape)
recharge_orders_enriched_df.head()

(1215, 22)


,metric_date,recharge_order_id,shopify_order_id,order_type,order_total,order_gross_revenue,order_tax,order_shipping,order_discounts,customer_id,order_id,shopify_customer_id,shopify_order_date,shopify_store,shopify_channel,shopify_first_product_category,shopify_revenue_excl_shipping_sgd,shopify_shipping_sgd,shopify_total_incl_shipping_sgd,shopify_discount_sgd,shopify_is_subscription_tag,shopify_source_file
0,2026-04-07,1306407778,6789789155583,recurring,62.1,62.1,0,0.0,0.0,237412247,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-07,1306407662,6789789057279,recurring,107.82,107.82,0,0.0,0.0,237898931,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-07,1306910933,6790149636351,checkout,128.01,142.23,0,0.0,-14.22,243992454,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-07,1307273448,6791342620927,checkout,126.08,141.14,0,0.0,-15.06,244069323,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-07,1306407554,6789789024511,recurring,33.07,33.07,0,0.0,0.0,175122062,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
